In [1]:
!pip install openpyxl

    PyYAML (>=5.1.*)
            ~~~~~~^


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
tf.config.run_functions_eagerly(True)

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Embedding, LSTM, SimpleRNN, GRU, Dense, TimeDistributed, Dropout, Bidirectional, Input
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from collections import defaultdict

import os

2026-01-30 10:48:13.747188: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 10:48:13.858385: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2026-01-30 10:48:13.858410: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-01-30 10:48:14.547300: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could 

In [3]:
#print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

In [4]:
#print(tf.test.is_gpu_available())  # Deprecated in TF 2.1
#print(tf.config.experimental.list_physical_devices('GPU'))  # Works for TF 2.x


In [5]:
# Force TensorFlow to use GPU only
#physical_devices = tf.config.list_physical_devices('GPU')
#if physical_devices:
#    tf.config.set_visible_devices(physical_devices[0], 'GPU')

In [6]:
# Allow memory growth
#gpus = tf.config.experimental.list_physical_devices('GPU')
#if gpus:
#    try:
#        for gpu in gpus:
#            tf.config.experimental.set_memory_growth(gpu, True)
#        print("GPU memory growth enabled")
#    except RuntimeError as e:
#        print(e)  # Memory growth must be set at program startup

In [7]:
## Use this for 1L sentences
folder_path = 'test_data'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

100

In [8]:
if "kon_agriculture_set1.txt" in txt_files:
    txt_files.remove("kon_agriculture_set1.txt")

print(len(txt_files))

99


In [9]:
taggedData = []
for file in txt_files:
    with open("test_data/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)
    
allTaggedData = [item for sublist in taggedData for item in sublist]
allTaggedData[:2]

['अस्वस्थतायेचीं\\N_NN कायम\\JJ प्रतिकां\\N_NN .\\RD_PUNC',
 'सगळ्या\\QT_QTF पयल्यान\\PR_PRP प्रशांतीची\\N_NNP उदका\\N_NN रेशा\\N_NN स्पश्ट\\RB जावन\\V_VM_VNF येता\\V_VAUX_VF .\\RD_PUNC']

In [10]:
with open('POS_tagged_data_20k.txt', 'r', encoding='utf-8') as f:
    sentences = f.readlines()

In [11]:
print(sentences[:5])
len(sentences)

['अवशेश\\JJ मोल\\N_NN .\\RD_PUNC\n', 'रिणाचे\\N_NN पयलीं\\N_NST .\\RD_PUNC\n', 'निर्देशीत\\JJ अर्थवेवस्था\\N_NN .\\RD_PUNC\n', 'अहिल्या\\N_NNP बाई\\N_NN .\\RD_PUNC\n', 'शीमशुल्काचे\\N_NN मुखेल\\JJ आयुक्त\\N_NN ,\\RD_PUNC गुजरात\\N_NNP .\\RD_PUNC\n']


20142

In [12]:
total_sentences = allTaggedData + sentences

In [13]:
len(total_sentences)

119233

In [14]:
allTaggedData = list(set(total_sentences))
len(allTaggedData)

118938

In [15]:
sentences = [sentence.strip().split() for sentence in allTaggedData]

In [16]:
# Extract words and tags
word_tag_pairs = [[token.rsplit("\\", 1) for token in sentence] for sentence in sentences]

In [17]:
# Create word and tag vocabularies
word_freq = defaultdict(int)
tag_freq = defaultdict(int)

In [18]:
filtered_sentences = []

for sentence in word_tag_pairs:
    filtered_sentence = [item for item in sentence if isinstance(item, (list, tuple)) and len(item) == 2]  # Keep only valid pairs
    
    if filtered_sentence:  # Add to filtered list if not empty
        filtered_sentences.append(filtered_sentence)

    for word, tag in filtered_sentence:
        word_freq[word] += 1
        tag_freq[tag] += 1

## Replace original list if needed
word_tag_pairs = filtered_sentences 

In [19]:
print(len(word_freq))
print(len(tag_freq))

167937
83


In [20]:
# Create word and tag mappings
word2idx = {word: idx + 2 for idx, word in enumerate(word_freq.keys())}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {idx: word for word, idx in word2idx.items()}

In [21]:
tag2idx = {tag: idx for idx, tag in enumerate(tag_freq.keys())}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

In [22]:
# Convert sentences into index sequences
X = [[word2idx.get(word, 1) for word, tag in sentence] for sentence in word_tag_pairs]
y = [[tag2idx[tag] for word, tag in sentence] for sentence in word_tag_pairs]

In [23]:
# Pad sequences
max_len = max(len(sentence) for sentence in X)
print(max_len)
X_padded = pad_sequences(X, maxlen=max_len, padding="post")
y_padded = pad_sequences(y, maxlen=max_len, padding="post")
y_categorical = [to_categorical(tag_seq, num_classes=len(tag2idx)) for tag_seq in y_padded]
y_categorical = np.array(y_categorical)

134


In [24]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_padded, y_categorical, test_size=0.03, random_state=42)

In [25]:
print(len(X_train))
len(X_test)

115075


3560

In [26]:
# Build Bidirectional LSTM Model
input_layer = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=len(word2idx), output_dim=128, mask_zero=True)(input_layer)
bi_gru_layer = Bidirectional(GRU(units=256, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(embedding_layer)
#lstm_layer = LSTM(units=256, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)(embedding_layer)
#bi_lstm_layer1 = Bidirectional(LSTM(units=256, return_sequences=True, dropout=0.3))(embedding_layer)
#bi_rnn_layer1 = SimpleRNN(units=256, return_sequences=True, dropout=0.3)(embedding_layer)
#bi_rnn_layer1 = Bidirectional(SimpleRNN(units=256, return_sequences=True, dropout=0.3))(embedding_layer)
#bi_gru_layer1 = GRU(units=256, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(embedding_layer)
#bi_gru_layer1 = Bidirectional(GRU(units=256, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(embedding_layer)
output_layer = TimeDistributed(Dense(len(tag2idx), activation="softmax"))(bi_gru_layer)

2026-01-29 16:17:56.530400: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2026-01-29 16:17:56.530642: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2026-01-29 16:17:56.530758: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/v

In [27]:
model = Model(input_layer, output_layer)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [28]:
# If using a custom function
@tf.function
def train_model():
    model.fit(X_train, y_train, batch_size=128, epochs=4, validation_data=(X_test, y_test))

train_model()  

Epoch 1/4


/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


900/900 [==============================] - 6343s 7s/step - loss: 0.6812 - accuracy: 0.8131 - val_loss: 0.3804 - val_accuracy: 0.8821
Epoch 2/4
900/900 [==============================] - 6339s 7s/step - loss: 0.2807 - accuracy: 0.9079 - val_loss: 0.3633 - val_accuracy: 0.8884
Epoch 3/4
900/900 [==============================] - 6350s 7s/step - loss: 0.2231 - accuracy: 0.9253 - val_loss: 0.3690 - val_accuracy: 0.8887
Epoch 4/4
900/900 [==============================] - 6338s 7s/step - loss: 0.1857 - accuracy: 0.9388 - val_loss: 0.3824 - val_accuracy: 0.8881


In [29]:
# Save the model
model.save("new_updated_biGru.h5")



In [30]:
import dill
with open("word2idx_bigru.pkl", "wb") as f:
    dill.dump(word2idx, f)


with open("idx2tag_bigru.pkl", "wb") as f:
    dill.dump(idx2tag, f)

In [2]:
import dill
# Load the word2idx object from the file
with open("word2idx_bigru.pkl", "rb") as f:
    word2idx1 = dill.load(f)

# Load the idx2tag object from the file
with open("idx2tag_bigru.pkl", "rb") as f:
    idx2tag1 = dill.load(f)

In [3]:
from keras.models import load_model
loaded_model = load_model("new_updated_biGru.h5")

2026-01-30 10:49:23.305285: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2026-01-30 10:49:23.305419: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/vidyaapati/boost/lib:/home/vidyaapati/boost/lib:/home/vidyaapati/zlib/lib:/home/vidyaapati/zlib/lib:
2026-01-30 10:49:23.305487: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/v

In [4]:
def pos_tag_sentence(sentence):
    tokens = sentence.split()
    token_ids = [word2idx1.get(token, 1) for token in tokens]
    token_ids_padded = pad_sequences([token_ids], maxlen=134, padding="post")
    predictions = loaded_model.predict(token_ids_padded)[0]
    predicted_tags = [idx2tag1[np.argmax(tag)] for tag in predictions][:len(tokens)]
    return list(zip(tokens, predicted_tags))

In [5]:
# Example usage
print(pos_tag_sentence("ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवूच घेना ."))

# [('ग्रेटर', 'N_NN'), ('नोयडा', 'N_NN'), ('वेस्टाचे', 'N_NN'), ('त्रास', 'N_NN'), ('कमी', 'QT_QTF'), ('जावपाचें', 'V_VM_VNF'), ('नांवूच', 'N_NN'), ('घेना', 'V_VM_VF'), ('.', 'RD_PUNC')]
#[('आयकलां', 'V_VM_VF'), ('?', 'RD_PUNC'), ('लागीं', 'N_NST'), ('ना', 'RP_NEG'), ('आमचो', 'PR_PRP'), ('गांव', 'N_NN'), ('.', 'RD_PUNC'), ('चलून', 'V_VM_VNF'), ('चलून', 'V_VM_VNF'), ('कितलीं', 'QT_QTF'), ('म्हणून', 'V_VM_VNF'), ('चलतलीं', 'N_NN'), ('आमी', 'PR_PRP'), ('?', 'RD_PUNC')]



/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 1s 1s/step
[('ग्रेटर', 'N_NNP'), ('नोयडा', 'N_NNP'), ('वेस्टाचे', 'N_NNP'), ('त्रास', 'N_NN'), ('कमी', 'QT_QTF'), ('जावपाचें', 'V_VM_VNF'), ('नांवूच', 'N_NN'), ('घेना', 'V_VM_VF'), ('.', 'RD_PUNC')]


In [6]:
df_anwani = pd.read_excel('ref_sentences/anwani_ref_sentences.xlsx', engine='openpyxl')
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_anwani = df_anwani.iloc[:, :3]
df_anwani = df_anwani.iloc[:100, :3]
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...


In [8]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(pos_tag_sentence)
df_anwani.head()

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 1s 1s/step


,sentences_cleaned,predicted_sentences,tagged_sentences,output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,"[(आयकलां, V_VM_VF), (?, RD_PUNC), (लागीं, N_NS..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,"[(दोन, QT_QTC), (तीन, QT_QTC), (दिसांनी, N_NN)..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,"[(आनी, CC_CCD), (जाली, V_VM_VNF), (ना, RP_NEG)..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,"[(हांव, PR_PRP), (अशें, DM_DMD), (आसा, V_VM_VF..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,"[(पूण, CC_CCS), (हांगा, N_NST), (रावनय, N_NN),..."


In [9]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

'सहस्त्रधारा\\N_NNP ही\\DM_DMD देहरादूनाची\\N_NNP पळोवपाची\\V_VM_VNF मुखेल\\JJ सुवात\\N_NN .\\RD_PUNC'

In [10]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...


In [11]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
print(mismatched_indices)

for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

[48, 49]

In [13]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [14]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,14,14,0.785714,0.714286,0.785714,0.738095,"[V_VM_VF, RD_PUNC, N_NST, RP_NEG, DM_DMD, N_NN...","[V_VM_VF, RD_PUNC, N_NST, RP_NEG, PR_PRP, N_NN..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,11,11,0.818182,0.909091,0.818182,0.854545,"[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_...","[QT_QTC, QT_QTC, N_NN, V_VM_VNF, JJ, N_NN, RD_..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,22,22,1.000000,1.000000,1.000000,1.000000,"[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR...","[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,17,17,0.882353,1.000000,0.882353,0.911765,"[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM...","[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...,15,15,0.933333,0.966667,0.933333,0.941414,"[CC_CCS, N_NST, V_VM_VNF, V_VM_VNF, RP_NEG, RD...","[CC_CCS, N_NST, N_NN, V_VM_VNF, RP_NEG, RD_PUN..."


In [15]:
df_anwani.to_excel("new_bi_gru_100_anwani.xlsx", index=False)

In [16]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        13
      CC_CCS       0.66      0.91      0.76        43
      DM_DMD       0.81      0.92      0.86        52
      DM_DMI       0.78      0.70      0.74        10
      DM_DMQ       1.00      0.33      0.50        15
      DM_DMR       0.00      0.00      0.00         0
          JJ       0.47      0.58      0.52        12
         NST       0.00      0.00      0.00         1
        N_NN       0.75      0.96      0.84       201
       N_NNP       0.00      0.00      0.00         1
       N_NST       0.89      0.91      0.90        78
      PR_PRF       1.00      1.00      1.00         1
      PR_PRI       1.00      0.45      0.62        11
      PR_PRP       0.97      0.90      0.93       131
      PR_PRQ       0.65      0.65      0.65        20
         PSP       0.78      0.55      0.65        38
      QT_QTC       0.94      0.94      0.94        16
  

In [18]:
df_ref = pd.read_excel('ref_sentences/ref_sentences.xlsx', engine='openpyxl')
df_ref.head()

,ref_sentence
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...


In [19]:
ref_sentences = df_ref['ref_sentence'].tolist()
print(len(ref_sentences))
ref_sentences = ref_sentences[:75]
print(len(ref_sentences))

107
75


In [20]:
df_test = pd.DataFrame(ref_sentences, columns=['actual_tags'])
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  75 non-null     object
dtypes: object(1)
memory usage: 728.0+ bytes


In [29]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
#df_test["sentence"], df_test["labels"] = zip(*df_test["actual_tags"].map(preprocess_sentence))
#df_test.info()

In [22]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)
            words.append(word)
            tags.append(tag)
    return words, tags

In [23]:
df_test[["sentence", "labels"]] = (
    df_test["actual_tags"]
    .apply(preprocess_sentence)
    .apply(pd.Series)
)

In [24]:
df_test.head()

,actual_tags,sentence,labels
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[सहस्त्रधारा, ही, देहरादूनाची, पळोवपाची, मुखेल...","[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_..."
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[खजियाराची, सांजवेळ, दर, भोंवडेकाराक, थाकाय, द...","[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]"
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[आधुनिकीकरण, आनी, नागरीकरणाक, लागून, जिणेशैलें...","[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN..."
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[हे, मागणे, प्रमाणें, ३१, ऑक्टोबर, ,, २०००चे, ...","[DM_DMD, N_NN, PSP, QT_QTC, N_NN, RD_PUNC, QT_..."
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[स्पोर्टिंगाक, सुरवातीक, गोल, करपाच्यो, कांय, ...","[N_NN, N_NST, N_NN, V_VM_VNF, QT_QTF, N_NN, PS..."


In [26]:
def joinSen(words):
    sentence = " ".join(words)
    return sentence

In [27]:
df_test["joined_sent"] = df_test["sentence"].apply(joinSen)
df_test.head()

,actual_tags,sentence,labels,joined_sent
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[सहस्त्रधारा, ही, देहरादूनाची, पळोवपाची, मुखेल...","[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_...",सहस्त्रधारा ही देहरादूनाची पळोवपाची मुखेल सुवात .
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[खजियाराची, सांजवेळ, दर, भोंवडेकाराक, थाकाय, द...","[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]",खजियाराची सांजवेळ दर भोंवडेकाराक थाकाय दिता .
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[आधुनिकीकरण, आनी, नागरीकरणाक, लागून, जिणेशैलें...","[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN...",आधुनिकीकरण आनी नागरीकरणाक लागून जिणेशैलेंत जाल...
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[हे, मागणे, प्रमाणें, ३१, ऑक्टोबर, ,, २०००चे, ...","[DM_DMD, N_NN, PSP, QT_QTC, N_NN, RD_PUNC, QT_...","हे मागणे प्रमाणें ३१ ऑक्टोबर , २०००चे मध्य रात..."
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[स्पोर्टिंगाक, सुरवातीक, गोल, करपाच्यो, कांय, ...","[N_NN, N_NST, N_NN, V_VM_VNF, QT_QTF, N_NN, PS...",स्पोर्टिंगाक सुरवातीक गोल करपाच्यो कांय संदी ल...


In [28]:
df_test["output"] = df_test["joined_sent"].apply(pos_tag_sentence)
df_test.head()

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 1s 1s/step


,actual_tags,sentence,labels,joined_sent,output
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[सहस्त्रधारा, ही, देहरादूनाची, पळोवपाची, मुखेल...","[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_...",सहस्त्रधारा ही देहरादूनाची पळोवपाची मुखेल सुवात .,"[(सहस्त्रधारा, N_NNP), (ही, DM_DMD), (देहरादून..."
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[खजियाराची, सांजवेळ, दर, भोंवडेकाराक, थाकाय, द...","[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]",खजियाराची सांजवेळ दर भोंवडेकाराक थाकाय दिता .,"[(खजियाराची, N_NNP), (सांजवेळ, N_NN), (दर, JJ)..."
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[आधुनिकीकरण, आनी, नागरीकरणाक, लागून, जिणेशैलें...","[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN...",आधुनिकीकरण आनी नागरीकरणाक लागून जिणेशैलेंत जाल...,"[(आधुनिकीकरण, N_NN), (आनी, CC_CCD), (नागरीकरणा..."
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[हे, मागणे, प्रमाणें, ३१, ऑक्टोबर, ,, २०००चे, ...","[DM_DMD, N_NN, PSP, QT_QTC, N_NN, RD_PUNC, QT_...","हे मागणे प्रमाणें ३१ ऑक्टोबर , २०००चे मध्य रात...","[(हे, DM_DMD), (मागणे, N_NN), (प्रमाणें, PSP),..."
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[स्पोर्टिंगाक, सुरवातीक, गोल, करपाच्यो, कांय, ...","[N_NN, N_NST, N_NN, V_VM_VNF, QT_QTF, N_NN, PS...",स्पोर्टिंगाक सुरवातीक गोल करपाच्यो कांय संदी ल...,"[(स्पोर्टिंगाक, N_NN), (सुरवातीक, N_NN), (गोल,..."


In [30]:
df_test["joined_output"] = df_test["output"].apply(makeTagedSentences)
df_test.drop(["output"], axis=1, inplace=True)
df_test.drop(["sentence"], axis=1, inplace=True)
df_test.head()

,actual_tags,labels,joined_sent,joined_output
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_...",सहस्त्रधारा ही देहरादूनाची पळोवपाची मुखेल सुवात .,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]",खजियाराची सांजवेळ दर भोंवडेकाराक थाकाय दिता .,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN...",आधुनिकीकरण आनी नागरीकरणाक लागून जिणेशैलेंत जाल...,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[DM_DMD, N_NN, PSP, QT_QTC, N_NN, RD_PUNC, QT_...","हे मागणे प्रमाणें ३१ ऑक्टोबर , २०००चे मध्य रात...",हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\N_NNP ऑक्...
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[N_NN, N_NST, N_NN, V_VM_VNF, QT_QTF, N_NN, PS...",स्पोर्टिंगाक सुरवातीक गोल करपाच्यो कांय संदी ल...,स्पोर्टिंगाक\N_NN सुरवातीक\N_NN गोल\N_NN करपाच...


In [32]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['actual_tags_length'] = df['actual_tags'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['actual_tags_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

df_test, mismatched_indices = add_length_columns(df_test)
print(mismatched_indices)

for ind in mismatched_indices:
    df_test = df_test.drop(ind)

df_test.info()

[]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   actual_tags         75 non-null     object
 1   labels              75 non-null     object
 2   joined_sent         75 non-null     object
 3   joined_output       75 non-null     object
 4   actual_tags_length  75 non-null     int64 
 5   joined_sent_length  75 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 3.6+ KB


In [34]:
# Apply function to each row
df_test[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_test.apply(lambda row: evaluate_tagging(row["actual_tags"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_test.head()

,actual_tags,labels,joined_sent,joined_output,actual_tags_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_...",सहस्त्रधारा ही देहरादूनाची पळोवपाची मुखेल सुवात .,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,7,7,1.000000,1.000000,1.000000,1.000000,"[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_...","[N_NNP, DM_DMD, N_NNP, V_VM_VNF, JJ, N_NN, RD_..."
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]",खजियाराची सांजवेळ दर भोंवडेकाराक थाकाय दिता .,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,7,7,1.000000,1.000000,1.000000,1.000000,"[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]","[N_NNP, N_NN, JJ, N_NN, N_NN, V_VM_VF, RD_PUNC]"
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN...",आधुनिकीकरण आनी नागरीकरणाक लागून जिणेशैलेंत जाल...,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,15,15,1.000000,1.000000,1.000000,1.000000,"[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN...","[N_NN, CC_CCD, N_NN, PSP, N_NN, V_VM_VNF, N_NN..."
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[DM_DMD, N_NN, PSP, QT_QTC, N_NN, RD_PUNC, QT_...","हे मागणे प्रमाणें ३१ ऑक्टोबर , २०००चे मध्य रात...",हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\N_NNP ऑक्...,20,20,0.850000,0.914286,0.850000,0.854978,"[DM_DMD, N_NN, PSP, QT_QTC, N_NN, RD_PUNC, QT_...","[DM_DMD, N_NN, PSP, N_NNP, N_NNP, RD_PUNC, QT_..."
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[N_NN, N_NST, N_NN, V_VM_VNF, QT_QTF, N_NN, PS...",स्पोर्टिंगाक सुरवातीक गोल करपाच्यो कांय संदी ल...,स्पोर्टिंगाक\N_NN सुरवातीक\N_NN गोल\N_NN करपाच...,23,23,0.956522,0.920290,0.956522,0.936759,"[N_NN, N_NST, N_NN, V_VM_VNF, QT_QTF, N_NN, PS...","[N_NN, N_NN, N_NN, V_VM_VNF, QT_QTF, N_NN, PSP..."


In [35]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_test["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_test["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        33
      CC_CCS       0.78      0.90      0.84        20
   CC_CCS_UT       0.00      0.00      0.00         1
       CC_UT       0.00      0.00      0.00         1
      DM_DMD       0.74      0.96      0.84        27
      DM_DMI       0.00      0.00      0.00         1
      DM_DMQ       0.00      0.00      0.00         2
      DM_DMR       1.00      0.60      0.75         5
          JJ       0.84      0.84      0.84        43
        N_NN       0.98      0.95      0.96       315
       N_NNP       0.92      0.96      0.94        91
       N_NST       0.92      0.76      0.83        29
      PR_PRF       1.00      1.00      1.00         4
      PR_PRI       0.80      0.80      0.80         5
      PR_PRL       1.00      1.00      1.00         5
      PR_PRP       0.97      0.91      0.94        33
      PR_PRQ       1.00      1.00      1.00         3
  

In [36]:
df_test.to_excel("bi_gru_75_ILCI.xlsx", index=False) 

In [37]:
df_mannual = pd.read_excel('ref_sentences/ref_sentences_Mannual.xlsx', engine='openpyxl')
df_mannual.head()

,sentences,ref_sentences,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
df_mannual = df_mannual.iloc[:100]
df_mannual = df_mannual.iloc[:, :2]

In [42]:
df_mannual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   sentences      100 non-null    object
 1   ref_sentences  100 non-null    object
dtypes: object(2)
memory usage: 1.7+ KB


In [43]:
df_mannual.head(2)

,sentences,ref_sentences
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...


In [44]:
df_mannual["output"] = df_mannual["sentences"].apply(pos_tag_sentence)
df_mannual.head()

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/tensorflow/python/data/ops/structured_function.py:256: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 1s 1s/step


,sentences,ref_sentences,output
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[(मनशाच्या, N_NN), (शरिरांत, N_NN), (206, QT_Q..."
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[(मनशाचें, N_NN), (काळीज, N_NN), (दिसाक, N_NN)..."
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[(सूर्य, N_NN), (पृथ्वी, N_NN), (परस, PSP), (3..."
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[(नील, N_NNP), (आर्मस्ट्राँग, N_NNP), (हो, DM_..."
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[(पृथ्वीचो, N_NN), (सगल्यांत, RP_INTF), (व्हडल..."


In [45]:
df_mannual["joined_output"] = df_mannual["output"].apply(makeTagedSentences)
df_mannual.drop(["output"], axis=1, inplace=True)
#df_mannual.drop(["sentence"], axis=1, inplace=True)
df_mannual.head()

,sentences,ref_sentences,joined_output
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\QT_QTC लाख\QT...
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\DM_DMD 1969\N_...
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,पृथ्वीचो\N_NN सगल्यांत\RP_INTF व्हडलो\JJ आनी\C...


In [46]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['ref_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

df_mannual, mismatched_indices = add_length_columns(df_mannual)
print(mismatched_indices)

for ind in mismatched_indices:
    df_mannual = df_mannual.drop(ind)

df_mannual.info()

[43, 55, 56, 75, 80, 92, 99]
<class 'pandas.core.frame.DataFrame'>
Int64Index: 93 entries, 0 to 98
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences             93 non-null     object
 1   ref_sentences         93 non-null     object
 2   joined_output         93 non-null     object
 3   ref_sentences_length  93 non-null     int64 
 4   joined_sent_length    93 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 4.4+ KB


In [47]:
# Apply function to each row
df_mannual[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_mannual.apply(lambda row: evaluate_tagging(row["ref_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_mannual.head()

,sentences,ref_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,6,6,1.000000,1.000000,1.000000,1.000000,"[N_NN, N_NN, QT_QTC, N_NN, V_VM_VF, RD_PUNC]","[N_NN, N_NN, QT_QTC, N_NN, V_VM_VF, RD_PUNC]"
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...,10,10,0.800000,0.850000,0.800000,0.812121,"[N_NN, N_NN, N_NN, RB, QT_QTC, N_NN, N_NN, N_N...","[N_NN, N_NN, N_NN, QT_QTF, QT_QTC, QT_QTC, N_N..."
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\QT_QTC लाख\QT...,20,20,0.700000,0.576190,0.700000,0.626667,"[N_NNP, N_NNP, PSP, QT_QTC, N_NN, PSP, QT_QTF,...","[N_NN, N_NN, PSP, QT_QTC, QT_QTC, PSP, QT_QTF,..."
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\DM_DMD 1969\N_...,12,12,0.916667,0.916667,0.916667,0.916667,"[N_NNP, N_NNP, PR_PRP, N_NN, N_NN, N_NNP, N_NN...","[N_NNP, N_NNP, DM_DMD, N_NN, N_NN, N_NNP, N_NN..."
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,पृथ्वीचो\N_NN सगल्यांत\RP_INTF व्हडलो\JJ आनी\C...,10,10,0.800000,0.733333,0.800000,0.750000,"[N_NNP, RP_INTF, JJ, CC_CCD, JJ, JJ, N_NN, CC_...","[N_NN, RP_INTF, JJ, CC_CCD, JJ, JJ, N_NN, CC_C..."


In [48]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_mannual["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_mannual["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        20
      CC_CCS       0.90      0.95      0.93        20
      DM_DMD       0.94      1.00      0.97        30
      DM_DMI       1.00      1.00      1.00         1
      DM_DMR       1.00      0.50      0.67         2
          JJ       0.91      0.87      0.89        60
        N_NN       0.96      0.94      0.95       345
       N_NNP       0.85      0.85      0.85        62
       N_NST       0.94      0.94      0.94        18
      PR_PRF       1.00      1.00      1.00         6
      PR_PRL       0.50      0.50      0.50         2
      PR_PRP       0.88      0.88      0.88         8
         PSP       0.97      0.94      0.95        32
      QT_QTC       0.93      1.00      0.97        28
      QT_QTF       0.83      1.00      0.91        20
      QT_QTO       0.50      1.00      0.67         3
          RB       0.83      0.77      0.80        13
  

In [49]:
df_mannual.to_excel("bi_gru_100_mannual.xlsx", index=False) 